# Batch model scoring with the ML App client

This notebook uploads 100,000 records to the existing scoring dataset family in the **Estates Sell Prices** Business Case and runs the latest published version of **Estates Sell Prices - AutoFEML - Scoring**. The backend resolves immutable input and model versions and stores a prediction dataset with complete lineage.

In [ ]:
BUSINESS_CASE_NAME = "Estates Sell Prices"
DATASET_NAME = "Estates Sell Prices - Scoring Data"
PIPELINE_NAME = "Estates Sell Prices - AutoFEML - Scoring"
INPUT_FILENAME = "estates-sale-prices-batch-scoring-100k.parquet"

## Client and authentication

Install the client from the repository root with `pip install -e .`. In automation, provide `ML_APP_API_URL` and `ML_APP_ACCESS_TOKEN` through a secret manager. For local interactive use, the notebook prompts for credentials when no token is configured.

In [ ]:
import getpass
import os
from pathlib import Path

from ml_app_client import MLAppClient

client = MLAppClient.from_env()
if not os.getenv("ML_APP_ACCESS_TOKEN", "").strip():
    client.login(input("Login: "), getpass.getpass("Password: "))

## Input file

The Parquet file is included in the repository. Path resolution supports running Jupyter from either the repository root or `examples/API-usage`.

In [ ]:
candidates = [
    Path.cwd() / "examples" / "data" / INPUT_FILENAME,
    Path.cwd().parent / "data" / INPUT_FILENAME,
]
scoring_file = next((path for path in candidates if path.is_file()), None)
if scoring_file is None:
    raise FileNotFoundError(
        f"Could not find {INPUT_FILENAME}; run Jupyter from the repository root or examples/API-usage"
    )
scoring_file

## Upload a new scoring dataset version

The client resolves the dataset attached to the Business Case and streams the Parquet file as a new immutable version. It does not materialize the 100,000 rows in notebook memory.

In [ ]:
uploaded = client.upload_dataset_version(
    scoring_file,
    business_case_name=BUSINESS_CASE_NAME,
    dataset_name=DATASET_NAME,
    description="Estates batch scoring input — 100k records",
    tags=["batch-scoring", "estates"],
)
rows = "unknown" if uploaded.row_count is None else f"{uploaded.row_count:,}"
print(
    f"Uploaded {uploaded.name} v{uploaded.version_number}; "
    f"dataset_id={uploaded.id}; rows={rows}"
)

## Start scoring with latest versions

The client selects the latest published pipeline version. Its dataset input uses the `latest` policy, so the backend binds the version uploaded above. With no `model_versions` override, the backend selects the latest model in the configured model family and binds its matching fitted Feature Engineering state.

In [ ]:
run = client.run_pipeline_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    pipeline_name=PIPELINE_NAME,
)
print(
    f"Started run_id={run.id}; pipeline_version_id={run.pipeline_version_id}; "
    f"status={run.status}"
)

## Alternative: a specific dataset version and model version

The commented example below scores a specific newly uploaded dataset version (`specific_dataset.id`) with a specific immutable model artifact. This published scoring pipeline currently uses the dataset policy `latest`, so the reproducible route is to upload the selected file as a new version and start the run immediately. `input_versions` does not override a `latest` policy; selecting an arbitrary older version requires a published pipeline version whose input policy is `select_at_run`. A model can be pinned directly with the scoring step key `scoring_1`.

In [ ]:
# SPECIFIC_INPUT_FILE = Path(r"C:\data\estates-scoring-v7.parquet")
# SPECIFIC_MODEL_ARTIFACT_ID = "00000000-0000-0000-0000-000000000000"  # ID from the model registry
#
# specific_dataset = client.upload_dataset_version(
#     SPECIFIC_INPUT_FILE,
#     business_case_name=BUSINESS_CASE_NAME,
#     dataset_name=DATASET_NAME,
#     description="Explicit dataset version for a reproducible scoring run",
# )
# pinned_run = client.run_pipeline_by_name(
#     business_case_name=BUSINESS_CASE_NAME,
#     pipeline_name=PIPELINE_NAME,
#     model_versions={"scoring_1": SPECIFIC_MODEL_ARTIFACT_ID},
# )
# print({
#     "dataset_version_id": specific_dataset.id,
#     "model_artifact_id": SPECIFIC_MODEL_ARTIFACT_ID,
#     "run_id": pinned_run.id,
# })

If a published pipeline version uses `select_at_run`, select an older dataset version with `input_versions={"de_1:scoring_input": SPECIFIC_DATASET_VERSION_ID}`. The map is intentionally omitted for the current `latest` policy.

## Wait for the result

Polling retrieves bounded run metadata only. The timeout does not cancel the platform job. A successful run creates an immutable prediction dataset with lineage to the input, model, fitted transform, and pipeline version.

In [ ]:
finished = client.wait_for_pipeline_run(
    run,
    timeout=3600,
    on_update=lambda current: print(
        f"status={current.status}; processed_rows={current.processed_row_count}"
    ),
)
rows = (
    "unknown"
    if finished.processed_row_count is None
    else f"{finished.processed_row_count:,}"
)
print(f"Completed: status={finished.status}; processed_rows={rows}")

## Inspect a bounded result preview

Resolve the prediction dataset from the run manifest and fetch only a small preview. This is suitable for interactive inspection and does not transfer the complete result to the notebook.

In [ ]:
prediction_dataset_id = client.prediction_dataset_id(finished)
preview = client.preview_dataset(prediction_dataset_id, limit=20)
print(
    f"Prediction dataset: {prediction_dataset_id}; "
    f"total_rows={preview['row_count']:,}; returned_rows={preview['returned_count']}"
)
preview["records"]

## Download the complete prediction dataset

The client streams the immutable Parquet result directly to disk in 1 MiB chunks. The complete dataset is never buffered in Python memory. Keep this file and its `prediction_dataset_id`: it can later be joined with actuals through a Monitoring pipeline, either in the application or through a separate integration workflow.

In [ ]:
output_directory = Path.cwd() / "scoring-results"
output_file = output_directory / f"estates-predictions-{finished.id}.parquet"
downloaded_file = client.download_dataset(prediction_dataset_id, output_file)
print(f"Saved {downloaded_file} ({downloaded_file.stat().st_size:,} bytes)")